In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
# !pip install duckdb

In [3]:
import duckdb
from pathlib import Path
import pandas as pd

In [4]:
PROJECT_ROOT = Path.cwd()

# Data directories
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
WAREHOUSE_DIR = DATA_DIR / "warehouse"

WAREHOUSE_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH = RAW_DIR / "CompaniesHouseData-2026-03-02.csv"
CSV_PATH = PROCESSED_DIR / "entity_master_v1.csv"
DUCKDB_PATH = WAREHOUSE_DIR / "project.duckdb"

print("CSV exists:", CSV_PATH.exists())
print("DuckDB path:", DUCKDB_PATH)

CSV exists: True
DuckDB path: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/data/warehouse/project.duckdb


In [5]:
duckdb_con = duckdb.connect(str(DUCKDB_PATH))
print("Connected to:", DUCKDB_PATH)

Connected to: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/data/warehouse/project.duckdb


In [6]:
duckdb_con.execute(f"""
    CREATE OR REPLACE TABLE entity_master_v1 AS
    SELECT *
    FROM read_csv_auto('{CSV_PATH}');
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
duckdb_con.execute("""
    SELECT *
    FROM entity_master_v1
    LIMIT 10
""").fetchdf()

,entity_id,entity_name,company_category,company_status,country_of_origin,incorporation_date,account_category,sic_text_1,sic_text_2,sic_text_3,sic_text_4,post_town,county,country,postcode,source_uri,entity_name_raw
0,16873705,FE NETWORK LIMITED,Private Limited Company,Active,United Kingdom,2025-11-25,NO ACCOUNTS FILED,86210 - General medical practice activities,None,None,None,"HARLOW, ESSEX",None,UNITED KINGDOM,CM20 1YS,http://business.data.gov.uk/id/company/16873705,!FE NETWORK LIMITED
1,15073164,NFLECTION ADVISORY LIMITED,Private Limited Company,Active,United Kingdom,2023-08-15,TOTAL EXEMPTION FULL,70229 - Management consultancy activities othe...,None,None,None,POTTERS BAR,HERTFORDSHIRE,ENGLAND,EN6 2DA,http://business.data.gov.uk/id/company/15073164,!NFLECTION ADVISORY LIMITED
2,13522064,NFOGENIE LTD,Private Limited Company,Active,United Kingdom,2021-07-21,MICRO ENTITY,58290 - Other software publishing,None,None,None,LONDON,GREATER LONDON,UNITED KINGDOM,WC2H 9JQ,http://business.data.gov.uk/id/company/13522064,!NFOGENIE LTD
3,11006939,NNOV8 LIMITED,Private Limited Company,Active,United Kingdom,2017-10-11,MICRO ENTITY,62090 - Other information technology service a...,70229 - Management consultancy activities othe...,None,None,EDENBRIDGE,None,ENGLAND,TN8 5NF,http://business.data.gov.uk/id/company/11006939,!NNOV8 LIMITED
4,SC606050,NSPIRED INVESTMENTS LTD,Private Limited Company,Active,United Kingdom,2018-08-22,TOTAL EXEMPTION FULL,68209 - Other letting and operating of own or ...,None,None,None,ABERDEEN,None,SCOTLAND,AB11 7SY,http://business.data.gov.uk/id/company/SC606050,!NSPIRED INVESTMENTS LTD
5,07687209,OBAC UK LIMITED,Private Limited Company,Active,United Kingdom,2011-06-29,TOTAL EXEMPTION FULL,70229 - Management consultancy activities othe...,None,None,None,TADLEY,HAMPSHIRE,None,RG26 5AT,http://business.data.gov.uk/id/company/07687209,!OBAC UK LIMITED
6,13310195,""" TRIPLE D"" PROPERTIES LIMITED",Private Limited Company,Active,United Kingdom,2021-04-01,UNAUDITED ABRIDGED,68209 - Other letting and operating of own or ...,68320 - Management of real estate on a fee or ...,None,None,BIRMINGHAM,None,ENGLAND,B24 9NB,http://business.data.gov.uk/id/company/13310195,""" TRIPLE D"" PROPERTIES LIMITED"
7,11303802,"""1ST RATE"" PSYCHOLOGY SERVICES LTD",Private Limited Company,Active,United Kingdom,2018-04-11,MICRO ENTITY,85600 - Educational support services,86900 - Other human health activities,None,None,GREAT WAKERING,ESSEX,UNITED KINGDOM,SS3 0GW,http://business.data.gov.uk/id/company/11303802,"""1ST RATE"" PSYCHOLOGY SERVICES LTD"
8,10694769,"""786"" MAZ OFFICE SUPPORT LIMITED",Private Limited Company,Active,United Kingdom,2017-03-28,MICRO ENTITY,82110 - Combined office administrative service...,None,None,None,PRESTON,None,ENGLAND,PR2 9QL,http://business.data.gov.uk/id/company/10694769,"""786"" MAZ OFFICE SUPPORT LIMITED"
9,15761044,"""A TASTE OF TUSCANY"" LTD",Private Limited Company,Active,United Kingdom,2024-06-04,NO ACCOUNTS FILED,56101 - Licensed restaurants,68209 - Other letting and operating of own or ...,None,None,LONDON,None,UNITED KINGDOM,WC2H 9JQ,http://business.data.gov.uk/id/company/15761044,"""A TASTE OF TUSCANY"" LTD"


In [8]:
duckdb_con.execute(f"""
CREATE OR REPLACE TABLE entity_mortgage_features AS
SELECT
    "CompanyNumber" AS entity_id,
    "Mortgages.NumMortCharges" AS num_mort_charges,
    "Mortgages.NumMortOutstanding" AS num_mort_outstanding,
    "Mortgages.NumMortPartSatisfied" AS num_mort_part_satisfied,
    "Mortgages.NumMortSatisfied" AS num_mort_satisfied
FROM read_csv_auto('{RAW_DATA_PATH}')
""")

In [9]:
duckdb_con.execute("""
    SELECT *
    FROM entity_mortgage_features
    LIMIT 10
""").fetchdf()

,entity_id,num_mort_charges,num_mort_outstanding,num_mort_part_satisfied,num_mort_satisfied
0,08209948,0,0,0,0
1,11743365,0,0,0,0
2,16873705,0,0,0,0
3,15073164,0,0,0,0
4,13522064,0,0,0,0
5,11006939,0,0,0,0
6,SC606050,5,5,0,0
7,SC421617,0,0,0,0
8,FC031362,0,0,0,0
9,07687209,1,0,0,1


In [10]:
duckdb_con.execute("""
CREATE OR REPLACE TABLE entity_signal_base AS
SELECT
    master.*,
    mortgage.num_mort_charges,
    mortgage.num_mort_outstanding,
    mortgage.num_mort_part_satisfied,
    mortgage.num_mort_satisfied
FROM entity_master_v1 master
LEFT JOIN entity_mortgage_features mortgage
    ON master.entity_id = mortgage.entity_id
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
duckdb_con.execute("""
    SELECT *
    FROM entity_signal_base
    LIMIT 10
""").fetchdf()

,entity_id,entity_name,company_category,company_status,country_of_origin,incorporation_date,account_category,sic_text_1,sic_text_2,sic_text_3,...,post_town,county,country,postcode,source_uri,entity_name_raw,num_mort_charges,num_mort_outstanding,num_mort_part_satisfied,num_mort_satisfied
0,14581880,ACORN SCIENTIFIC MARKETING LTD,Private Limited Company,Active,United Kingdom,2023-01-10,TOTAL EXEMPTION FULL,73110 - Advertising agencies,None,None,...,LYTHAM ST. ANNES,None,ENGLAND,FY8 2RR,http://business.data.gov.uk/id/company/14581880,ACORN SCIENTIFIC MARKETING LTD,0,0,0,0
1,SC835604,ACORN SCOTLAND LTD,Private Limited Company,Active,United Kingdom,2025-01-28,NO ACCOUNTS FILED,87900 - Other residential care activities n.e.c.,None,None,...,PAISLEY,None,SCOTLAND,PA3 1QJ,http://business.data.gov.uk/id/company/SC835604,ACORN SCOTLAND LTD,0,0,0,0
2,02232508,ACORN SCREEN PRODUCTS LIMITED,Private Limited Company,Active,United Kingdom,1988-03-18,MICRO ENTITY,20302 - Manufacture of printing ink,None,None,...,LOUGHBOROUGH,None,ENGLAND,LE11 1JP,http://business.data.gov.uk/id/company/02232508,ACORN SCREEN PRODUCTS LIMITED,5,0,0,5
3,12125709,ACORN SEATING LTD,Private Limited Company,Active,United Kingdom,2019-07-27,MICRO ENTITY,13921 - Manufacture of soft furnishings,None,None,...,KETTERING,NORTHAMPTONSHIRE,ENGLAND,NN15 6WJ,http://business.data.gov.uk/id/company/12125709,ACORN SEATING LTD,0,0,0,0
4,15795093,ACORN SECURITY AND FM SOLUTIONS LTD,Private Limited Company,Active,United Kingdom,2024-06-22,NO ACCOUNTS FILED,96090 - Other service activities n.e.c.,None,None,...,SOUTHAMPTON,None,ENGLAND,SO14 0AZ,http://business.data.gov.uk/id/company/15795093,ACORN SECURITY AND FM SOLUTIONS LTD,0,0,0,0
5,03814979,ACORN SECURITY LOCKSMITHS LIMITED,Private Limited Company,Active,United Kingdom,1999-07-27,MICRO ENTITY,80200 - Security systems service activities,None,None,...,LONDON,None,ENGLAND,N12 0NL,http://business.data.gov.uk/id/company/03814979,ACORN SECURITY LOCKSMITHS LIMITED,1,1,0,0
6,06776336,ACORN SEEDS LIMITED,Private Limited Company,Active,United Kingdom,2008-12-18,TOTAL EXEMPTION FULL,01610 - Support activities for crop production,None,None,...,DOWNHAM MARKET,NORFOLK,None,PE38 0DX,http://business.data.gov.uk/id/company/06776336,ACORN SEEDS LIMITED,1,1,0,0
7,11550423,ACORN SERVICE SOLUTIONS LTD,Private Limited Company,Active,United Kingdom,2018-09-04,TOTAL EXEMPTION FULL,71122 - Engineering related scientific and tec...,None,None,...,CHESTERFIELD,None,ENGLAND,S43 4JE,http://business.data.gov.uk/id/company/11550423,ACORN SERVICE SOLUTIONS LTD,0,0,0,0
8,SC201254,ACORN SERVICES (EDINBURGH) LIMITED,Private Limited Company,Active,United Kingdom,1999-11-03,TOTAL EXEMPTION FULL,43999 - Other specialised construction activit...,None,None,...,EDINBURGH,None,SCOTLAND,EH15 2AT,http://business.data.gov.uk/id/company/SC201254,ACORN SERVICES (EDINBURGH) LIMITED,0,0,0,0
9,08206271,ACORN SERVICES (HERTS) LIMITED,Private Limited Company,Active,United Kingdom,2012-09-07,MICRO ENTITY,01610 - Support activities for crop production,01630 - Post-harvest crop activities,None,...,WELWYN GARDEN CITY,None,ENGLAND,AL7 1TW,http://business.data.gov.uk/id/company/08206271,ACORN SERVICES (HERTS) LIMITED,0,0,0,0


In [12]:
duckdb_con.execute("""
CREATE OR REPLACE TABLE entity_risk_signals_v1 AS
SELECT
    entity_id,
    entity_name,
    post_town,
    postcode,
    country,
    company_category,        
    company_status,
    incorporation_date,
    account_category,
    num_mort_charges,
    num_mort_outstanding,
    num_mort_part_satisfied,
    num_mort_satisfied,
    sic_text_1,

    CASE
        WHEN incorporation_date >= current_date - INTERVAL 365 DAY
        THEN 1 ELSE 0
    END AS new_entity_flag,


    CASE
        WHEN post_town IS NULL and postcode IS NULL
        THEN 1 ELSE 0
    END AS missing_location_flag,


    CASE
        WHEN account_category = 'NO ACCOUNTS FILED'
        THEN 1 ELSE 0
    END AS no_accounts_filed_flag,                 


    CASE
        WHEN coalesce(num_mort_outstanding, 0) > 0
        THEN 1 ELSE 0
    END AS has_outstanding_mortgage_flag,

                   
    CASE
        WHEN coalesce(num_mort_outstanding, 0) > 0
        AND (
            coalesce(num_mort_part_satisfied, 0) > 0
            OR coalesce(num_mort_satisfied, 0) > 0
        )
        THEN 1 ELSE 0
    END AS mixed_mortgage_profile_flag,

FROM entity_signal_base
""")

In [13]:
duckdb_con.execute("""
    SELECT * 
    FROM entity_risk_signals_v1
    LIMIT 10
""").fetchdf()

,entity_id,entity_name,post_town,postcode,country,company_category,company_status,incorporation_date,account_category,num_mort_charges,num_mort_outstanding,num_mort_part_satisfied,num_mort_satisfied,sic_text_1,new_entity_flag,missing_location_flag,no_accounts_filed_flag,has_outstanding_mortgage_flag,mixed_mortgage_profile_flag
0,14581880,ACORN SCIENTIFIC MARKETING LTD,LYTHAM ST. ANNES,FY8 2RR,ENGLAND,Private Limited Company,Active,2023-01-10,TOTAL EXEMPTION FULL,0,0,0,0,73110 - Advertising agencies,0,0,0,0,0
1,SC835604,ACORN SCOTLAND LTD,PAISLEY,PA3 1QJ,SCOTLAND,Private Limited Company,Active,2025-01-28,NO ACCOUNTS FILED,0,0,0,0,87900 - Other residential care activities n.e.c.,0,0,1,0,0
2,02232508,ACORN SCREEN PRODUCTS LIMITED,LOUGHBOROUGH,LE11 1JP,ENGLAND,Private Limited Company,Active,1988-03-18,MICRO ENTITY,5,0,0,5,20302 - Manufacture of printing ink,0,0,0,0,0
3,12125709,ACORN SEATING LTD,KETTERING,NN15 6WJ,ENGLAND,Private Limited Company,Active,2019-07-27,MICRO ENTITY,0,0,0,0,13921 - Manufacture of soft furnishings,0,0,0,0,0
4,15795093,ACORN SECURITY AND FM SOLUTIONS LTD,SOUTHAMPTON,SO14 0AZ,ENGLAND,Private Limited Company,Active,2024-06-22,NO ACCOUNTS FILED,0,0,0,0,96090 - Other service activities n.e.c.,0,0,1,0,0
5,03814979,ACORN SECURITY LOCKSMITHS LIMITED,LONDON,N12 0NL,ENGLAND,Private Limited Company,Active,1999-07-27,MICRO ENTITY,1,1,0,0,80200 - Security systems service activities,0,0,0,1,0
6,06776336,ACORN SEEDS LIMITED,DOWNHAM MARKET,PE38 0DX,None,Private Limited Company,Active,2008-12-18,TOTAL EXEMPTION FULL,1,1,0,0,01610 - Support activities for crop production,0,0,0,1,0
7,11550423,ACORN SERVICE SOLUTIONS LTD,CHESTERFIELD,S43 4JE,ENGLAND,Private Limited Company,Active,2018-09-04,TOTAL EXEMPTION FULL,0,0,0,0,71122 - Engineering related scientific and tec...,0,0,0,0,0
8,SC201254,ACORN SERVICES (EDINBURGH) LIMITED,EDINBURGH,EH15 2AT,SCOTLAND,Private Limited Company,Active,1999-11-03,TOTAL EXEMPTION FULL,0,0,0,0,43999 - Other specialised construction activit...,0,0,0,0,0
9,08206271,ACORN SERVICES (HERTS) LIMITED,WELWYN GARDEN CITY,AL7 1TW,ENGLAND,Private Limited Company,Active,2012-09-07,MICRO ENTITY,0,0,0,0,01610 - Support activities for crop production,0,0,0,0,0


In [14]:
duckdb_con.execute("""
CREATE OR REPLACE TABLE entity_risk_signals_v2 AS
SELECT
    *,
    new_entity_flag
    + missing_location_flag
    + no_accounts_filed_flag
    + has_outstanding_mortgage_flag
    + mixed_mortgage_profile_flag
    AS review_priority_score,

    CASE
        WHEN (
            new_entity_flag
            + missing_location_flag
            + no_accounts_filed_flag
            + has_outstanding_mortgage_flag
            + mixed_mortgage_profile_flag
        ) >= 4 THEN 'High'
        
        WHEN (
            new_entity_flag
            + missing_location_flag
            + no_accounts_filed_flag
            + has_outstanding_mortgage_flag
            + mixed_mortgage_profile_flag
        ) = 3 THEN 'Medium'
        
        ELSE 'Low'
    END AS review_priority_band

FROM entity_risk_signals_v1
""")

In [18]:
duckdb_con.execute("""
    SELECT review_priority_band, COUNT(*) AS total_number
    FROM entity_risk_signals_v2
    GROUP BY review_priority_band
    ORDER BY total_number DESC
""").fetchdf()

,review_priority_band,total_number
0,Low,4175779
1,Medium,15558
2,High,198


In [20]:
duckdb_con.execute("""
    SELECT
        entity_id,
        entity_name,
        incorporation_date,
        account_category,
        num_mort_charges,
        num_mort_outstanding,
        sic_text_1,
        new_entity_flag,
        missing_location_flag,
        no_accounts_filed_flag,
        has_outstanding_mortgage_flag,
        mixed_mortgage_profile_flag,
        review_priority_score,
        review_priority_band
    FROM entity_risk_signals_v2
    WHERE review_priority_band = 'High'
    ORDER BY review_priority_score DESC, entity_name
""").fetchdf()

,entity_id,entity_name,incorporation_date,account_category,num_mort_charges,num_mort_outstanding,sic_text_1,new_entity_flag,missing_location_flag,no_accounts_filed_flag,has_outstanding_mortgage_flag,mixed_mortgage_profile_flag,review_priority_score,review_priority_band
0,16502889,... AND RELAX COTTAGES LTD,2025-06-07,NO ACCOUNTS FILED,2,1,68209 - Other letting and operating of own or ...,1,0,1,1,1,4,High
1,16435051,360SQUARE LTD,2025-05-07,NO ACCOUNTS FILED,3,1,68100 - Buying and selling of own real estate,1,0,1,1,1,4,High
2,16531177,50-52 CR SUBCO LIMITED,2025-06-20,NO ACCOUNTS FILED,3,2,41100 - Development of building projects,1,0,1,1,1,4,High
3,16392049,AD3 PROPERTIES LTD,2025-04-16,NO ACCOUNTS FILED,3,2,68209 - Other letting and operating of own or ...,1,0,1,1,1,4,High
4,16484242,ALOEHAWK LIMITED,2025-05-30,NO ACCOUNTS FILED,6,3,68209 - Other letting and operating of own or ...,1,0,1,1,1,4,High
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193,16537084,WOOD GREEN DEVELOPERS LTD,2025-06-23,NO ACCOUNTS FILED,3,2,41100 - Development of building projects,1,0,1,1,1,4,High
194,16517513,WP 29 PROPERTY LTD,2025-06-13,NO ACCOUNTS FILED,3,2,68100 - Buying and selling of own real estate,1,0,1,1,1,4,High
195,16508805,XELINIA PROPERTIES UK M1 LIMITED,2025-06-10,NO ACCOUNTS FILED,2,1,68100 - Buying and selling of own real estate,1,0,1,1,1,4,High
196,16440236,ZALO ESTATES LIMITED,2025-05-09,NO ACCOUNTS FILED,2,1,68100 - Buying and selling of own real estate,1,0,1,1,1,4,High


In [21]:
duckdb_con.execute("""
    COPY entity_risk_signals_v2
    TO 'data/processed/entity_risk_signals_v2.csv'
    (HEADER, DELIMITER ',');
""")

In [22]:
duckdb_con.close()
print("DuckDB connection closed.")

DuckDB connection closed.
